<a href="https://colab.research.google.com/github/AyazN/DLS-final-project/blob/develop/notebooks/full_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [1]:
%cd /content

!git clone https://github.com/AyazN/DLS-final-project.git

%cd /content/DLS-final-project

/content
Cloning into 'DLS-final-project'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 173 (delta 72), reused 127 (delta 49), pack-reused 0 (from 0)
Receiving objects: 100% (173/173), 69.98 KiB | 1.04 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/content/DLS-final-project


In [3]:
!pip install -q -r requirements.txt
!pip install -q -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 57.4 MB/s eta 0:00:00
ERROR: file:///content/DLS-final-project does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [4]:
import sys

PROJECT_ROOT = "/content/DLS-final-project"

sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, f"{PROJECT_ROOT}/src")

In [5]:
from aise.contracts import Query, SearchDocument, SearchResult

print("Project imports successfully")

Project imports successfully


In [7]:
from pathlib import Path
import shutil

drive_archive = Path(
    "/content/drive/MyDrive/Inno/DLS/AISE/embeddings/"
    "minilm_embeddings_600k.tar"
)

local_archive = Path("/content/minilm_embeddings_600k.tar")

if not local_archive.exists():
    shutil.copy2(drive_archive, local_archive)

print(local_archive)

/content/minilm_embeddings_600k.tar


In [8]:
%cd /content/DLS-final-project

!tar -xf /content/minilm_embeddings_600k.tar

/content/DLS-final-project


In [9]:
from pathlib import Path

EMBEDDING_DIR = Path(
    "data/processed/embeddings/"
    "sentence-transformers__all-MiniLM-L6-v2"
)

required_files = [
    "embeddings.npy",
    "ids.npy",
    "metadata.parquet",
    "run_config.json",
]

for filename in required_files:
    path = EMBEDDING_DIR / filename
    print(filename, path.exists(), path)

embeddings.npy True data/processed/embeddings/sentence-transformers__all-MiniLM-L6-v2/embeddings.npy
ids.npy True data/processed/embeddings/sentence-transformers__all-MiniLM-L6-v2/ids.npy
metadata.parquet True data/processed/embeddings/sentence-transformers__all-MiniLM-L6-v2/metadata.parquet
run_config.json True data/processed/embeddings/sentence-transformers__all-MiniLM-L6-v2/run_config.json


In [11]:
import json
import numpy as np
import pandas as pd

embeddings = np.load(
    EMBEDDING_DIR / "embeddings.npy",
    mmap_mode="r",
)

ids = np.load(
    EMBEDDING_DIR / "ids.npy",
    allow_pickle=True,
)

metadata = pd.read_parquet(
    EMBEDDING_DIR / "metadata.parquet"
)

with open(EMBEDDING_DIR / "run_config.json", encoding="utf-8") as file:
    run_config = json.load(file)

print("Embeddings:", embeddings.shape, embeddings.dtype)
print("IDs:", ids.shape)
print("Metadata:", metadata.shape)
print("Encoder:", run_config)

Embeddings: (600000, 384) float32
IDs: (600000,)
Metadata: (600000, 10)
Encoder: {'created_at_utc': '2026-07-11T21:30:10.150543+00:00', 'status': 'complete', 'input_rows': 600000, 'input_columns': ['model_id', 'likes', 'downloads', 'tags', 'pipeline_tag', 'library_name', 'createdAt', 'languages', 'modelCard'], 'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'model_dir': '/home/dbayazitov/Desktop/project/DLS-final-project/data/processed/embeddings/sentence-transformers__all-MiniLM-L6-v2', 'embedding_file': '/home/dbayazitov/Desktop/project/DLS-final-project/data/processed/embeddings/sentence-transformers__all-MiniLM-L6-v2/embeddings.npy', 'ids_file': '/home/dbayazitov/Desktop/project/DLS-final-project/data/processed/embeddings/sentence-transformers__all-MiniLM-L6-v2/ids.npy', 'metadata_file': '/home/dbayazitov/Desktop/project/DLS-final-project/data/processed/embeddings/sentence-transformers__all-MiniLM-L6-v2/metadata.parquet', 'progress_file': '/home/dbayazitov/Desktop/project/D

In [12]:
assert embeddings.shape[0] == len(ids) == len(metadata)
assert embeddings.shape[1] == 384
assert embeddings.dtype == np.float32

In [13]:
import faiss
import numpy as np
from pathlib import Path

INDEX_PATH = Path("data/indexes/minilm_flat_ip.faiss")
INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)

if INDEX_PATH.exists():
    index = faiss.read_index(str(INDEX_PATH))
else:
    index = faiss.IndexFlatIP(embeddings.shape[1])

    batch_size = 50_000

    for start in range(0, len(embeddings), batch_size):
        stop = min(start + batch_size, len(embeddings))

        batch = np.asarray(
            embeddings[start:stop],
            dtype=np.float32,
        )

        index.add(batch)
        print(f"Indexed {stop:,}/{len(embeddings):,}")

    faiss.write_index(index, str(INDEX_PATH))

print("Vectors in index:", index.ntotal)

Indexed 50,000/600,000
Indexed 100,000/600,000
Indexed 150,000/600,000
Indexed 200,000/600,000
Indexed 250,000/600,000
Indexed 300,000/600,000
Indexed 350,000/600,000
Indexed 400,000/600,000
Indexed 450,000/600,000
Indexed 500,000/600,000
Indexed 550,000/600,000
Indexed 600,000/600,000
Vectors in index: 600000


In [14]:
assert index.ntotal == 600_000
assert index.d == 384

In [15]:
DRIVE_INDEX = Path(
    "/content/drive/MyDrive/Inno/DLS/AISE/indices/"
    "minilm_flat_ip.faiss"
)

if not DRIVE_INDEX.exists():
    shutil.copy2(INDEX_PATH, DRIVE_INDEX)

In [16]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
def dense_search(query_text: str, top_k: int = 10):
    query_vector = encoder.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    scores, positions = index.search(query_vector, top_k)

    rows = []

    for rank, (score, position) in enumerate(
        zip(scores[0], positions[0]),
        start=1,
    ):
        row = metadata.iloc[int(position)]

        rows.append({
            "rank": rank,
            "score": float(score),
            "model_id": str(ids[position]),
            "title": row.get("title", row.get("name", "")),
        })

    return pd.DataFrame(rows)

In [19]:
dense_search("football teacher", top_k=10)

,rank,score,model_id,title
0,1,0.378699,OpenRL/tizero,
1,2,0.363234,irfanananda28/learningunsloth,
2,3,0.340789,CoreyMorris/poca-SoccerTwos-football-is-life,
3,4,0.330136,IsaacMwesigwa/footballer-recognition-6,
4,5,0.327941,matheusgeda/poca-SoccerTwos,
5,6,0.327279,jonathansculley/poca-SoccerTwos,
6,7,0.325894,qbbian/poca-SoccerTwos,
7,8,0.325757,GeorgeImmanuel/poco_soccer_two,
8,9,0.324802,JulioSnchezD/poca-SoccerTwos,
9,10,0.323819,brettgoehre/poca-SoccerTwosMemory,
